# XGBoost FROM SCRATCH (Tanpa Library XGBoost) — Skenario NoTune

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import os, pickle, time, warnings
warnings.filterwarnings('ignore')

# Library XGBoost HANYA dipakai sebagai pembanding (baseline), TIDAK dipakai di implementasi from-scratch
import xgboost as xgb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report
)

print('✅ Library siap')


✅ Library siap


## 2. Konfigurasi Path & Skenario (sama persis dengan notebook modelling Anda)

In [2]:
BASE_DIR      = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
SPLIT_DIR     = os.path.join(BASE_DIR, 'data', 'splits', 'splits_v3')
RESULT_DIR    = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULT_DIR, exist_ok=True)

SEED        = 42
LABEL_MAP   = {'keluhan': 0, 'saran': 1, 'pujian': 2}
INV_MAP     = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES = ['keluhan', 'saran', 'pujian']

K_A_MANUAL   = 2000
K_A_GABUNGAN = 8000
K_B_GABUNGAN = 10000

SKENARIO = [
    {'nama': 'A-Manual',   'tr': 'v2b_manual_train_balanced.csv', 'te': 'v2b_manual_test.csv', 'k': K_A_MANUAL},
    {'nama': 'A-Gabungan', 'tr': 'v2b_agab_train_balanced.csv',   'te': 'v2b_agab_test.csv',   'k': K_A_GABUNGAN},
    {'nama': 'B-Gabungan', 'tr': 'v2b_bgab_train_balanced.csv',   'te': 'v2b_bgab_test.csv',   'k': K_B_GABUNGAN},
]

print('Skenario yang akan diuji (NoTune saja):')
for s in SKENARIO:
    print(f'  {s["nama"]:12s} k={s["k"]:,}')


Skenario yang akan diuji (NoTune saja):
  A-Manual     k=2,000
  A-Gabungan   k=8,000
  B-Gabungan   k=10,000


## 3. Konfigurasi Khusus Implementasi *From-Scratch*


In [3]:
N_ESTIMATORS_SCRATCH = 300     # samakan dengan library (300) kalau Anda punya waktu cukup;
                                # turunkan (mis. 100) untuk demo lebih cepat -- tren hasil tetap terlihat
N_BINS               = 32      # jumlah bin histogram per fitur (mirip prinsip tree_method='hist' pada XGBoost asli)
MAX_DEPTH            = 6
LEARNING_RATE        = 0.1
SUBSAMPLE            = 0.8
COLSAMPLE_BYTREE     = 0.8
REG_LAMBDA           = 1.0     # default XGBoost (reg_lambda)
GAMMA                = 0.0     # default XGBoost (min_split_loss)
MIN_CHILD_WEIGHT     = 1.0     # default XGBoost

# k_best KHUSUS untuk versi scratch (lebih kecil dari versi library) agar waktu training wajar.
# Berdasarkan uji coba kami: k=2.000 (n~6rb baris, ~13 menit/300 iterasi) dan k=1.500 pada skenario
# gabungan (n~19-23rb baris, ~30-35 menit/300 iterasi) di CPU standar. Set None untuk memakai k yang
# SAMA PERSIS dengan versi library (Bagian 2) -- tapi siap-siap waktu training jauh lebih lama
# (k=8.000-10.000 pada puluhan ribu baris bisa memakan berjam-jam karena murni Python/NumPy).
K_BEST_SCRATCH_OVERRIDE = {'A-Manual': 2000, 'A-Gabungan': 1500, 'B-Gabungan': 1500}

print(f'n_estimators (scratch) = {N_ESTIMATORS_SCRATCH}')
print(f'n_bins histogram       = {N_BINS}')


n_estimators (scratch) = 300
n_bins histogram       = 32


## 4. Fungsi Fitur & Evaluasi (tetap pakai scikit-learn, sesuai instruksi)

In [4]:
def build_features(train_texts, train_labels, k_best, max_features=100000):
    tfidf = TfidfVectorizer(
        ngram_range=(1, 2), max_features=max_features,
        sublinear_tf=True, min_df=2,
        token_pattern=r'[a-zA-Z_][a-zA-Z_]+',
    )
    X = tfidf.fit_transform(train_texts)
    y_enc = np.array([LABEL_MAP[l] for l in train_labels])
    sel = SelectKBest(chi2, k=min(k_best, X.shape[1]))
    X_sel = sel.fit_transform(X, y_enc)
    print(f'  TF-IDF: {X.shape[1]:,} -> Chi2 k={k_best:,} -> {X_sel.shape[1]:,} fitur')
    return tfidf, sel, X_sel, y_enc


def get_xy(skenario, split_dir):
    df_tr = pd.read_csv(os.path.join(split_dir, skenario['tr']))
    df_te = pd.read_csv(os.path.join(split_dir, skenario['te']))
    X_tr = df_tr['text_v2'].fillna('').values
    y_tr = df_tr['label_pks'].values
    X_te = df_te['text_v2'].fillna('').values
    y_te = np.array([LABEL_MAP[l] for l in df_te['label_pks'].values])
    print(f'  Train (balanced): {len(df_tr):,} | Test (asli): {len(df_te):,}')
    return X_tr, y_tr, X_te, y_te


def get_sw(y_train):
    sw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    return np.array([sw[y] for y in y_train])


def evaluate(y_test, y_pred, nama):
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1s = f1_score(y_test, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    prec = precision_score(y_test, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    rec = recall_score(y_test, y_pred, average=None, labels=[0, 1, 2], zero_division=0)

    print(f'\n{"="*60}\nHASIL -- {nama}\n{"="*60}')
    print(f'  Accuracy   : {acc*100:.2f}%')
    print(f'  Macro F1   : {f1:.4f}')
    print(f'  F1 Keluhan : {f1s[0]:.4f}  Prec:{prec[0]:.4f}  Rec:{rec[0]:.4f}')
    print(f'  F1 Saran   : {f1s[1]:.4f}  Prec:{prec[1]:.4f}  Rec:{rec[1]:.4f}')
    print(f'  F1 Pujian  : {f1s[2]:.4f}  Prec:{prec[2]:.4f}  Rec:{rec[2]:.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))
    return {'nama': nama, 'acc': acc*100, 'f1': f1, 'f1_k': f1s[0], 'f1_s': f1s[1], 'f1_p': f1s[2]}

print('✅ Fungsi fitur & evaluasi siap (scikit-learn)')


✅ Fungsi fitur & evaluasi siap (scikit-learn)


## 5. Implementasi XGBoost DARI NOL (NumPy)

In [5]:
class _Node:
    __slots__ = ['is_leaf', 'feature', 'threshold', 'threshold_value', 'left', 'right', 'value']
    def __init__(self):
        self.is_leaf = True
        self.feature = None
        self.threshold = None
        self.threshold_value = None
        self.left = None
        self.right = None
        self.value = 0.0


def _histogram(bin_idx_sub, g_sub, h_sub, n_feat_sub, n_bins, chunk_size=800):
    """Histogram gradien & hessian per (fitur, bin), divektorisasi lewat trik bincount.
    Fitur diproses per-batch (chunk_size) supaya penggunaan memori tetap terkendali
    walau jumlah fitur (k_best) besar (mis. 8.000-10.000)."""
    m = bin_idx_sub.shape[0]
    hist_g = np.empty((n_feat_sub, n_bins))
    hist_h = np.empty((n_feat_sub, n_bins))
    for start in range(0, n_feat_sub, chunk_size):
        end = min(start + chunk_size, n_feat_sub)
        width = end - start
        offsets = (np.arange(width) * n_bins)[None, :]
        combined = (bin_idx_sub[:, start:end] + offsets).ravel()
        hist_g[start:end] = np.bincount(combined, weights=np.repeat(g_sub, width), minlength=width * n_bins).reshape(width, n_bins)
        hist_h[start:end] = np.bincount(combined, weights=np.repeat(h_sub, width), minlength=width * n_bins).reshape(width, n_bins)
    return hist_g, hist_h


def _best_split(hist_g, hist_h, G_tot, H_tot, lam, gamma, min_child_weight):
    """Persamaan (10): Gain = 1/2[GL^2/(HL+lam) + GR^2/(HR+lam) - (GL+GR)^2/(HL+HR+lam)] - gamma"""
    cum_g = np.cumsum(hist_g, axis=1)
    cum_h = np.cumsum(hist_h, axis=1)
    GL = cum_g[:, :-1]
    HL = cum_h[:, :-1]
    GR = G_tot - GL
    HR = H_tot - HL
    valid = (HL >= min_child_weight) & (HR >= min_child_weight)
    gain = 0.5 * ((GL**2)/(HL+lam) + (GR**2)/(HR+lam) - (G_tot**2)/(H_tot+lam)) - gamma
    gain = np.where(valid, gain, -np.inf)
    if not np.isfinite(gain).any():
        return None
    flat = np.argmax(gain)
    fi, bi = np.unravel_index(flat, gain.shape)
    best_gain = gain[fi, bi]
    if best_gain <= 1e-12:
        return None
    return fi, bi, best_gain


def _grow_tree(X_bin, g, h, row_idx, feat_subset, n_bins, depth, max_depth,
               lam, gamma, min_child_weight, min_samples_leaf=5):
    node = _Node()
    G_tot, H_tot = g[row_idx].sum(), h[row_idx].sum()

    if depth >= max_depth or len(row_idx) < 2*min_samples_leaf or H_tot <= 1e-12:
        node.value = -G_tot / (H_tot + lam)
        return node

    bin_idx_sub = X_bin[np.ix_(row_idx, feat_subset)]
    hist_g, hist_h = _histogram(bin_idx_sub, g[row_idx], h[row_idx], len(feat_subset), n_bins)
    split = _best_split(hist_g, hist_h, G_tot, H_tot, lam, gamma, min_child_weight)

    if split is None:
        node.value = -G_tot / (H_tot + lam)
        return node

    fi, bi, gain = split
    feat = feat_subset[fi]
    col_vals_bin = X_bin[row_idx, feat]
    left_mask = col_vals_bin <= bi
    left_idx, right_idx = row_idx[left_mask], row_idx[~left_mask]

    if len(left_idx) < min_samples_leaf or len(right_idx) < min_samples_leaf:
        node.value = -G_tot / (H_tot + lam)
        return node

    node.is_leaf = False
    node.feature = feat
    node.threshold = bi   # indeks bin (dipetakan balik ke nilai asli lewat bin_edges saat prediksi)
    node.left  = _grow_tree(X_bin, g, h, left_idx,  feat_subset, n_bins, depth+1, max_depth, lam, gamma, min_child_weight, min_samples_leaf)
    node.right = _grow_tree(X_bin, g, h, right_idx, feat_subset, n_bins, depth+1, max_depth, lam, gamma, min_child_weight, min_samples_leaf)
    return node


def _predict_tree_bin(node, X_bin, idx):
    """Prediksi rekursif satu pohon untuk sekumpulan baris (pakai bin index, dipakai saat training)."""
    out = np.zeros(len(idx))
    if node.is_leaf:
        out[:] = node.value
        return out
    col = X_bin[idx, node.feature]
    left_mask = col <= node.threshold
    if left_mask.any():
        out[left_mask] = _predict_tree_bin(node.left, X_bin, idx[left_mask])
    if (~left_mask).any():
        out[~left_mask] = _predict_tree_bin(node.right, X_bin, idx[~left_mask])
    return out


def _predict_tree_raw(node, X):
    """Prediksi satu pohon untuk data BARU (nilai fitur asli, mis. Data Test), pakai bin_edges."""
    n = X.shape[0]
    out = np.zeros(n)
    def rec(node, idx):
        if node.is_leaf:
            out[idx] = node.value
            return
        left_mask = X[idx, node.feature] <= node.threshold_value
        rec(node.left,  idx[left_mask])
        rec(node.right, idx[~left_mask])
    rec(node, np.arange(n))
    return out


class XGBoostScratch:
    """Multi-class gradient boosted trees (mengikuti objective XGBoost 'multi:softmax'),
    diimplementasikan murni dengan NumPy -- TANPA library xgboost."""

    def __init__(self, n_classes=3, n_estimators=300, max_depth=6, learning_rate=0.1,
                 subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, gamma=0.0,
                 min_child_weight=1.0, n_bins=32, random_state=42):
        self.n_classes = n_classes
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.lr = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.min_child_weight = min_child_weight
        self.n_bins = n_bins
        self.rng = np.random.RandomState(random_state)
        self.trees_ = []          # list per boosting round -> list of n_classes trees
        self.bin_edges_ = None    # (n_features, n_bins-1)

    def _fit_bins(self, X):
        n_features = X.shape[1]
        edges = np.zeros((n_features, self.n_bins - 1))
        qs = np.linspace(0, 100, self.n_bins + 1)[1:-1]
        for f in range(n_features):
            col = X[:, f]
            e = np.percentile(col, qs)
            e = np.unique(e)
            if len(e) < self.n_bins - 1:
                e = np.pad(e, (0, self.n_bins - 1 - len(e)), constant_values=e[-1] if len(e) else 0.0)
            edges[f] = e
        return edges

    def _digitize(self, X):
        n_samples, n_features = X.shape
        X_bin = np.zeros((n_samples, n_features), dtype=np.int32)
        for f in range(n_features):
            X_bin[:, f] = np.searchsorted(self.bin_edges_[f], X[:, f], side='right')
        return np.clip(X_bin, 0, self.n_bins - 1)

    @staticmethod
    def _softmax(Z):
        Z = Z - Z.max(axis=1, keepdims=True)
        expz = np.exp(Z)
        return expz / expz.sum(axis=1, keepdims=True)

    def fit(self, X, y, sample_weight=None, verbose_every=25):
        X = np.asarray(X, dtype=np.float64)
        n_samples, n_features = X.shape
        if sample_weight is None:
            sample_weight = np.ones(n_samples)

        print(f'  Membangun histogram bin ({self.n_bins} bin x {n_features:,} fitur)...')
        self.bin_edges_ = self._fit_bins(X)
        X_bin = self._digitize(X)

        Y_onehot = np.eye(self.n_classes)[y]
        F = np.zeros((n_samples, self.n_classes))  # raw score akumulatif (Persamaan 13/14)
        n_sub_rows = max(1, int(round(self.subsample * n_samples)))
        n_sub_feats = max(1, int(round(self.colsample_bytree * n_features)))

        t0 = time.time()
        for it in range(self.n_estimators):
            P = self._softmax(F)  # Persamaan (7)
            round_trees = []
            row_sample = self.rng.choice(n_samples, size=n_sub_rows, replace=False)

            for c in range(self.n_classes):
                g = (P[:, c] - Y_onehot[:, c]) * sample_weight   # Persamaan (8)
                h = (P[:, c] * (1 - P[:, c])) * sample_weight    # Persamaan (9)

                feat_sample = self.rng.choice(n_features, size=n_sub_feats, replace=False)
                tree = _grow_tree(X_bin, g, h, row_sample, feat_sample, self.n_bins,
                                   depth=0, max_depth=self.max_depth,
                                   lam=self.reg_lambda, gamma=self.gamma,
                                   min_child_weight=self.min_child_weight)
                # tempelkan nilai threshold ASLI (bukan indeks bin) supaya bisa dipakai untuk data baru
                self._attach_raw_threshold(tree, feat_sample)

                pred_all = _predict_tree_bin(tree, X_bin, np.arange(n_samples))  # Persamaan (13): f_k(x)
                F[:, c] += self.lr * pred_all
                round_trees.append(tree)

            self.trees_.append(round_trees)

            if (it + 1) % verbose_every == 0 or it == self.n_estimators - 1:
                elapsed = time.time() - t0
                eta = elapsed / (it + 1) * (self.n_estimators - it - 1)
                print(f'    iterasi {it+1:>4}/{self.n_estimators}  |  waktu: {elapsed:6.1f}s  |  perkiraan sisa: {eta:6.1f}s')

        return self

    def _attach_raw_threshold(self, node, feat_subset_unused=None):
        if node.is_leaf:
            return
        node.threshold_value = self.bin_edges_[node.feature][node.threshold] if node.threshold < self.n_bins - 1 else np.inf
        self._attach_raw_threshold(node.left)
        self._attach_raw_threshold(node.right)

    def predict_raw(self, X):
        X = np.asarray(X, dtype=np.float64)
        n_samples = X.shape[0]
        F = np.zeros((n_samples, self.n_classes))
        for round_trees in self.trees_:
            for c, tree in enumerate(round_trees):
                F[:, c] += self.lr * _predict_tree_raw(tree, X)
        return F

    def predict_proba(self, X):
        return self._softmax(self.predict_raw(X))

    def predict(self, X):
        return np.argmax(self.predict_raw(X), axis=1)


print("✅ Kelas XGBoostScratch siap (murni NumPy, tanpa 'import xgboost' di dalam implementasinya)")


✅ Kelas XGBoostScratch siap (murni NumPy, tanpa 'import xgboost' di dalam implementasinya)


## 6. Jalankan XGBoost LIBRARY (baseline pembanding, NoTune, 3 skenario)

In [6]:
hasil_library = []

for s in SKENARIO:
    nama = f'XGB-Library-NoTune-{s["nama"]}'
    print(f'\n{"─"*60}\nSKENARIO: {nama} (k={s["k"]:,})\n{"─"*60}')

    X_tr, y_tr, X_te, y_te = get_xy(s, SPLIT_DIR)
    tfidf, sel, X_train, y_train = build_features(X_tr, y_tr, s['k'])
    X_test = sel.transform(tfidf.transform(X_te))
    sw_arr = get_sw(y_train)

    model = xgb.XGBClassifier(
        objective='multi:softmax', num_class=3, eval_metric='mlogloss',
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        device='cpu', random_state=SEED, n_jobs=-1, use_label_encoder=False,
    )
    t0 = time.time()
    model.fit(X_train, y_train, sample_weight=sw_arr, verbose=False)
    print(f'  Training (library): {time.time()-t0:.1f}s')

    hasil = evaluate(y_te, model.predict(X_test), nama)
    hasil_library.append(hasil)

print('\n✅ XGBoost LIBRARY (NoTune, 3 skenario) selesai!')



────────────────────────────────────────────────────────────
SKENARIO: XGB-Library-NoTune-A-Manual (k=2,000)
────────────────────────────────────────────────────────────
  Train (balanced): 6,118 | Test (asli): 1,530
  TF-IDF: 6,668 -> Chi2 k=2,000 -> 2,000 fitur
  Training (library): 3.7s

HASIL -- XGB-Library-NoTune-A-Manual
  Accuracy   : 77.78%
  Macro F1   : 0.7819
  F1 Keluhan : 0.7558  Prec:0.7707  Rec:0.7414
  F1 Saran   : 0.7339  Prec:0.7043  Rec:0.7661
  F1 Pujian  : 0.8562  Prec:0.8795  Rec:0.8341

              precision    recall  f1-score   support

     keluhan     0.7707    0.7414    0.7558       553
       saran     0.7043    0.7661    0.7339       513
      pujian     0.8795    0.8341    0.8562       464

    accuracy                         0.7778      1530
   macro avg     0.7848    0.7805    0.7819      1530
weighted avg     0.7814    0.7778    0.7789      1530


────────────────────────────────────────────────────────────
SKENARIO: XGB-Library-NoTune-A-Gabungan (

## 7. Jalankan XGBoost FROM SCRATCH (NoTune, 3 skenario)

In [7]:
hasil_scratch = []

for s in SKENARIO:
    nama = f'XGB-Scratch-NoTune-{s["nama"]}'
    k_used = (K_BEST_SCRATCH_OVERRIDE[s['nama']] if K_BEST_SCRATCH_OVERRIDE and s['nama'] in K_BEST_SCRATCH_OVERRIDE else s['k'])
    print(f'\n{"─"*60}\nSKENARIO: {nama} (k={k_used:,}, n_estimators={N_ESTIMATORS_SCRATCH})\n{"─"*60}')

    X_tr, y_tr, X_te, y_te = get_xy(s, SPLIT_DIR)
    tfidf, sel, X_train_sparse, y_train = build_features(X_tr, y_tr, k_used)
    X_test_sparse = sel.transform(tfidf.transform(X_te))

    # XGBoostScratch butuh array padat (dense) -- aman karena k sudah diperkecil lewat Chi-Square
    X_train = np.asarray(X_train_sparse.todense())
    X_test  = np.asarray(X_test_sparse.todense())
    sw_arr  = get_sw(y_train)

    model_scratch = XGBoostScratch(
        n_classes=3, n_estimators=N_ESTIMATORS_SCRATCH, max_depth=MAX_DEPTH,
        learning_rate=LEARNING_RATE, subsample=SUBSAMPLE, colsample_bytree=COLSAMPLE_BYTREE,
        reg_lambda=REG_LAMBDA, gamma=GAMMA, min_child_weight=MIN_CHILD_WEIGHT,
        n_bins=N_BINS, random_state=SEED,
    )
    t0 = time.time()
    model_scratch.fit(X_train, y_train, sample_weight=sw_arr, verbose_every=25)
    print(f'  Training (scratch): {time.time()-t0:.1f}s')

    hasil = evaluate(y_te, model_scratch.predict(X_test), nama)
    hasil_scratch.append(hasil)

    with open(os.path.join(RESULT_DIR, f'xgb_scratch_{s["nama"].replace("-","_")}.pkl'), 'wb') as f:
        pickle.dump(model_scratch, f)

print('\n✅ XGBoost FROM SCRATCH (NoTune, 3 skenario) selesai!')



────────────────────────────────────────────────────────────
SKENARIO: XGB-Scratch-NoTune-A-Manual (k=2,000, n_estimators=300)
────────────────────────────────────────────────────────────
  Train (balanced): 6,118 | Test (asli): 1,530
  TF-IDF: 6,668 -> Chi2 k=2,000 -> 2,000 fitur
  Membangun histogram bin (32 bin x 2,000 fitur)...
    iterasi   25/300  |  waktu:   96.5s  |  perkiraan sisa: 1061.5s
    iterasi   50/300  |  waktu:  171.8s  |  perkiraan sisa:  859.1s
    iterasi   75/300  |  waktu:  239.7s  |  perkiraan sisa:  719.1s
    iterasi  100/300  |  waktu:  305.9s  |  perkiraan sisa:  611.8s
    iterasi  125/300  |  waktu:  372.6s  |  perkiraan sisa:  521.6s
    iterasi  150/300  |  waktu:  438.3s  |  perkiraan sisa:  438.3s
    iterasi  175/300  |  waktu:  505.3s  |  perkiraan sisa:  360.9s
    iterasi  200/300  |  waktu:  571.9s  |  perkiraan sisa:  285.9s
    iterasi  225/300  |  waktu:  637.9s  |  perkiraan sisa:  212.6s
    iterasi  250/300  |  waktu:  704.6s  |  perkiraan

## 8. Perbandingan Hasil: Library vs From-Scratch


In [8]:
df_lib = pd.DataFrame(hasil_library)
df_scr = pd.DataFrame(hasil_scratch)

df_lib['Skenario'] = [n.replace('XGB-Library-NoTune-', '') for n in df_lib['nama']]
df_scr['Skenario'] = [n.replace('XGB-Scratch-NoTune-', '') for n in df_scr['nama']]

banding = df_lib[['Skenario','acc','f1']].merge(df_scr[['Skenario','acc','f1']], on='Skenario', suffixes=(' (Library)', ' (Scratch)'))
banding['Selisih Accuracy'] = banding['acc (Scratch)'] - banding['acc (Library)']
banding['Selisih Macro F1'] = banding['f1 (Scratch)'] - banding['f1 (Library)']

banding = banding.rename(columns={
    'acc (Library)':'Accuracy (Library)', 'f1 (Library)':'Macro F1 (Library)',
    'acc (Scratch)':'Accuracy (Scratch)', 'f1 (Scratch)':'Macro F1 (Scratch)',
})
print(banding.to_string(index=False))
banding


  Skenario  Accuracy (Library)  Macro F1 (Library)  Accuracy (Scratch)  Macro F1 (Scratch)  Selisih Accuracy  Selisih Macro F1
  A-Manual           77.777778            0.781950           63.071895            0.638243        -14.705882         -0.143706
A-Gabungan           82.375952            0.823380           65.431336            0.657000        -16.944616         -0.166380
B-Gabungan           83.725490            0.837415           65.163399            0.654964        -18.562092         -0.182451


,Skenario,Accuracy (Library),Macro F1 (Library),Accuracy (Scratch),Macro F1 (Scratch),Selisih Accuracy,Selisih Macro F1
0,A-Manual,77.777778,0.781950,63.071895,0.638243,-14.705882,-0.143706
1,A-Gabungan,82.375952,0.823380,65.431336,0.657000,-16.944616,-0.166380
2,B-Gabungan,83.725490,0.837415,65.163399,0.654964,-18.562092,-0.182451
